# NLT Week 2 — Speech Recognition Lab

You will record **your own voice** and transcribe it with two current models that sit on opposite
sides of the architectural divide:

| Model | Backbone | Open ASR Leaderboard | What it demonstrates |
|---|---|---|---|
| `openai/whisper-large-v3-turbo` | **attention encoder–decoder** | ~7.75 WER, RTFx 216 | punctuation, casing, 99 languages — and hallucination |
| `nvidia/parakeet-tdt-0.6b-v3` | FastConformer + **TDT transducer** | 6.34 WER, RTFx 3333 | better accuracy, ~15× the throughput, cannot hallucinate |

Then you measure **Word Error Rate** for both, and you break them with noise.

> **Runtime → Change runtime type → T4 GPU** before you start.

## 0. Setup

This upgrades `transformers` — Colab will ask you to restart the session. **Restart, then skip this cell and carry on from section 1.**

In [ ]:
%pip install -q -U "transformers>=5.13.0" jiwer librosa ffmpeg-python matplotlib
print("Installed. If Colab shows a RESTART SESSION button, click it, then continue from section 1.")

In [ ]:
import torch, transformers
print("transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - go to Runtime > Change runtime type > T4 GPU")
DEVICE = 0 if torch.cuda.is_available() else -1
DTYPE  = torch.float16 if torch.cuda.is_available() else torch.float32

## 1. Record yourself

Run the cell, press the button, read the sentence below aloud, press again to stop.

> *"The TU Dublin campus at Grangegorman hosts the School of Computer Science,
> where anaphoric reference and word error rate are discussed at some length."*

That sentence is deliberately awkward: proper nouns, a place name most models have never seen,
and two pieces of module jargon. Everything that follows runs on **your** audio — there is no
pre-canned sample to hide behind.

In [ ]:
from IPython.display import HTML, Audio, display
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np, io, ffmpeg

AUDIO_HTML = """
<script>
var my_div = document.createElement("DIV");
var my_btn = document.createElement("BUTTON");
my_btn.appendChild(document.createTextNode("Press to start recording"));
my_div.appendChild(my_btn);
document.body.appendChild(my_div);

var base64data = 0, reader, recorder, gumStream;
var recordButton = my_btn;

var handleSuccess = function(stream) {
  gumStream = stream;
  recorder = new MediaRecorder(stream);
  recorder.ondataavailable = function(e) {
    var url = URL.createObjectURL(e.data);
    var preview = document.createElement('audio');
    preview.controls = true; preview.src = url;
    document.body.appendChild(preview);
    reader = new FileReader();
    reader.readAsDataURL(e.data);
    reader.onloadend = function() { base64data = reader.result; }
  };
  recorder.start();
};

recordButton.innerText = "Recording... press to stop";
navigator.mediaDevices.getUserMedia({audio: true}).then(handleSuccess);

function toggleRecording() {
  if (recorder && recorder.state == "recording") {
      recorder.stop();
      gumStream.getAudioTracks()[0].stop();
      recordButton.innerText = "Saving the recording... pls wait!";
  }
}
function sleep(ms) { return new Promise(resolve => setTimeout(resolve, ms)); }

var data = new Promise(resolve=>{
  recordButton.onclick = ()=>{ toggleRecording(); sleep(2000).then(() => resolve(base64data.toString())); }
});
</script>
"""

def record_audio(path="/content/recording.wav"):
    display(HTML(AUDIO_HTML))
    data = eval_js("data")
    binary = b64decode(data.split(',')[1])
    process = (ffmpeg
        .input('pipe:0')
        .output('pipe:1', format='wav', ac=1, ar=16000)
        .run_async(pipe_stdin=True, pipe_stdout=True, pipe_stderr=True, quiet=True, overwrite_output=True))
    output, err = process.communicate(input=binary)
    # patch the RIFF chunk size, which ffmpeg cannot know when writing to a pipe
    q, b = len(output) - 8, []
    for _ in range(4):
        q, r = divmod(q, 256); b.append(r)
    riff = output[:4] + bytes(b) + output[8:]
    with open(path, "wb") as f:
        f.write(riff)
    return path

AUDIO_PATH = record_audio()
print("saved to", AUDIO_PATH)

In [ ]:
# Type in exactly what you said, including any stumbles. This is your ground truth for WER.
REFERENCE = ("The TU Dublin campus at Grangegorman hosts the School of "
             "Computer Science, where anaphoric reference and word error rate are discussed at some length.")

In [ ]:
import librosa
from IPython.display import Audio, display

SR = 16000
audio, _ = librosa.load(AUDIO_PATH, sr=SR, mono=True)   # both models expect 16 kHz mono
print(f"{len(audio)/SR:.1f} s of audio")
display(Audio(audio, rate=SR))
print("REFERENCE:", REFERENCE)

## 2. A harness so both models are measured the same way

`RESULTS` collects the transcript, the wall-clock time and the **RTFx** (audio seconds transcribed
per second of compute) for each model. RTFx is the number the leaderboards quote — higher is faster.

In [ ]:
import time, gc, torch
RESULTS = {}

def run(name, fn, audio=audio):
    """Time a transcription function and record the result."""
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = time.perf_counter()
    text = fn(audio)
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    dt = time.perf_counter() - t0
    RESULTS[name] = {"text": text, "seconds": dt, "rtfx": (len(audio)/SR)/dt}
    print(f"{name}\n  {dt:.2f}s   RTFx {(len(audio)/SR)/dt:.1f}\n  {text}\n")
    return text

def free():
    """Colab's free tier has ~15 GB of VRAM. Drop each model before loading the next."""
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 3. Model A — attention encoder–decoder (`whisper-large-v3-turbo`)

Encoder over a 30-second log-Mel window, autoregressive decoder cross-attending over all of it.
The decoder *is* a language model, so you get punctuation, casing and number formatting for free —
and 99 languages from one checkpoint.

`turbo` keeps the large-v3 encoder but distils the decoder from 32 layers down to 4: ~7× faster,
~0.3 WER worse.

The cost of that decoder: it is **non-monotonic**. Nothing forces its output to stay aligned with
the audio, which is what makes section 7 possible.

In [ ]:
from transformers import pipeline

asr = pipeline("automatic-speech-recognition",
               model="openai/whisper-large-v3-turbo",
               dtype=DTYPE, device=DEVICE,
               chunk_length_s=30)      # needed for anything longer than 30 s
run("whisper-large-v3-turbo (AED)", lambda a: asr(a.copy())["text"])

del asr; free()

## 4. Model B — TDT transducer (`parakeet-tdt-0.6b-v3`)

FastConformer encoder, Token-and-Duration Transducer decoder. The transducer is monotonic — it
walks forward through the audio and cannot jump backwards or invent text out of nothing — and the
*duration* head lets it skip frames instead of emitting a decision for every one.

0.6B parameters (a quarter of Whisper large-v3), 25 European languages, and a **better** mean WER
at roughly fifteen times the throughput. Compare its RTFx with Whisper's above: that is the whole
architectural argument in one number.

In [ ]:
asr = pipeline("automatic-speech-recognition",
               model="nvidia/parakeet-tdt-0.6b-v3",
               dtype=DTYPE, device=DEVICE)
run("parakeet-tdt-0.6b-v3 (TDT)", lambda a: asr(a.copy())["text"])

del asr; free()

## 5. Word Error Rate — and why normalisation is half the battle

$$WER = \frac{S + D + I}{S + D + C}$$

Run it twice: once on the raw strings, once after Whisper-style normalisation (lowercase, strip
punctuation, expand numbers, drop fillers). **Every WER figure you see on a leaderboard is the
second number.** The gap between the two columns is how much of "accuracy" is really formatting.

In [ ]:
import jiwer, pandas as pd
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer

norm = EnglishTextNormalizer({})

rows = []
for name, r in RESULTS.items():
    raw   = jiwer.wer(REFERENCE, r["text"])
    normd = jiwer.wer(norm(REFERENCE), norm(r["text"]))
    cer   = jiwer.cer(norm(REFERENCE), norm(r["text"]))
    rows.append({"model": name,
                 "WER raw": f"{raw*100:.1f}%",
                 "WER normalised": f"{normd*100:.1f}%",
                 "CER": f"{cer*100:.1f}%",
                 "seconds": f"{r['seconds']:.2f}",
                 "RTFx": f"{r['rtfx']:.0f}"})

pd.DataFrame(rows).set_index("model")

In [ ]:
# Where exactly did the errors happen? jiwer will align the two strings and show you.
for name, r in RESULTS.items():
    print("#", name)
    print(jiwer.visualize_alignment(
        jiwer.process_words(norm(REFERENCE), norm(r["text"]))))

## 6. Breaking them: a noise sweep

Clean read English is solved. The interesting question is how each architecture *degrades*.

Mix white noise in at decreasing SNR and plot WER. Watch for two different failure modes: the
transducer starts dropping and mangling words, while the attention decoder stays fluent and begins
producing confident, well-formed sentences that were never spoken.

In [ ]:
import numpy as np

def add_noise(a, snr_db):
    noise = np.random.randn(len(a)).astype(np.float32)
    a_pow, n_pow = np.mean(a**2), np.mean(noise**2)
    scale = np.sqrt(a_pow / (n_pow * (10 ** (snr_db / 10))))
    return (a + scale * noise).astype(np.float32)

SNRS = [30, 20, 10, 5, 0]
MODELS = {
    "whisper-turbo (AED)": "openai/whisper-large-v3-turbo",
    "parakeet (TDT)":      "nvidia/parakeet-tdt-0.6b-v3",
}

curves = {}
for label, mid in MODELS.items():
    kw = {"chunk_length_s": 30} if "whisper" in mid else {}
    p = pipeline("automatic-speech-recognition", model=mid, dtype=DTYPE, device=DEVICE, **kw)
    curve = []
    for snr in SNRS:
        hyp = p(add_noise(audio, snr))["text"]
        curve.append(jiwer.wer(norm(REFERENCE), norm(hyp)) * 100)
        print(f"{label:22s} SNR {snr:>2} dB  WER {curve[-1]:5.1f}%   {hyp[:70]}")
    curves[label] = curve
    del p; free()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))
for label, c in curves.items():
    plt.plot(SNRS, c, marker="o", label=label)
plt.gca().invert_xaxis()
plt.xlabel("SNR (dB)  — noisier to the right"); plt.ylabel("WER (%)")
plt.title("Degradation under additive noise"); plt.legend(); plt.grid(alpha=.3)
plt.show()

## TODO


1. **Jargon.** Record a second take containing five technical terms from this module. Count *entity*
   errors rather than total words. Does the ranking of the two models change?
2. **Normalisation.** Report WER before and after normalisation for both models. Which one benefits
   more, and what does that tell you about what each was trained to produce?
3. **Cost.** All of the above ran free. Look up the per-hour price of the equivalent commercial APIs
   (Deepgram Nova-3, OpenAI `gpt-4o-transcribe`, Google Chirp 3) and work out at what audio volume a
   self-hosted Parakeet becomes cheaper than an API call.
4. **Extending this:** the harness is model-agnostic. Adding a third system is one line in `MODELS` —
> `facebook/wav2vec2-base-960h` for the 2020 CTC baseline (no punctuation, no casing, no language
> model), or `Qwen/Qwen3-ASR-0.6B-hf` for a 2026 speech-LLM.